<a href="https://colab.research.google.com/github/abdullahminhas/cric-squad-selector/blob/main/cric_squad_selector_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building our player-match dataset from Cricsheet.
Tests, ODIs, and T20Is — one CSV each, committed to GitHub as we go.

In [1]:
# connect to GitHub repo
from google.colab import userdata
import os

GH_TOKEN = userdata.get('GH_TOKEN')
REPO_URL = f"https://{GH_TOKEN}@github.com/abdullahminhas/cric-squad-selector.git"

if not os.path.exists('cric-squad-selector'):
    !git clone {REPO_URL}

%cd cric-squad-selector
!git config user.email "aminhas1996@gmail.com"
!git config user.name "abdullahminhas"
os.makedirs('data', exist_ok=True)

print(os.getcwd())

Cloning into 'cric-squad-selector'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 61 (delta 25), reused 32 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 4.48 MiB | 5.95 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/cric-squad-selector
/content/cric-squad-selector


The function below downloads a Cricsheet zip, unzips it, and turns every match into player-level rows.
We check the download/unzip actually worked, so it fails loudly instead of silently.

In [2]:
# parser function
import json, pandas as pd

def build_player_match_dataset(zip_url, extract_folder):
    zip_name = extract_folder + ".zip"

    exit_code = os.system(f'wget -q --user-agent="Mozilla/5.0" {zip_url} -O {zip_name}')
    if exit_code != 0 or not os.path.exists(zip_name) or os.path.getsize(zip_name) == 0:
        raise RuntimeError(f"Download failed for {zip_url}")

    exit_code = os.system(f"unzip -q -o {zip_name} -d {extract_folder}")
    if exit_code != 0 or not os.path.exists(extract_folder):
        raise RuntimeError(f"Unzip failed for {zip_name}")

    all_rows = []

    for file in os.listdir(extract_folder):
        if not file.endswith(".json"):
            continue

        with open(os.path.join(extract_folder, file)) as f:
            data = json.load(f)

        info = data["info"]
        match_id = file.replace(".json", "")
        date = info["dates"][0]
        fmt = info.get("match_type")
        venue = info.get("venue")
        teams = info["teams"]
        winner = info["outcome"].get("winner")
        pom_list = info.get("player_of_match", [])
        registry = info.get("registry", {}).get("people", {})

        players_stats = {}
        for team, plist in info.get("players", {}).items():
            opp = teams[1] if team == teams[0] else teams[0]
            for p in plist:
                players_stats[p] = {
                    "match_id": match_id, "player_id": registry.get(p), "player_name": p,
                    "team": team, "opposition": opp, "date": date, "format": fmt, "venue": venue,
                    "batting_position": None, "runs": 0, "balls_faced": 0, "fours": 0, "sixes": 0,
                    "dismissal_type": None, "dismissed": False,
                    "balls_bowled": 0, "maidens": 0, "runs_conceded": 0, "wickets": 0,
                    "wides": 0, "no_balls": 0, "dot_balls": 0, "fours_conceded": 0, "sixes_conceded": 0,
                    "catches": 0, "run_outs": 0, "stumpings": 0,
                    "team_won": (winner == team) if winner else None,
                    "player_of_match": p in pom_list,
                }

        for innings in data.get("innings", []):
            bat_position_counter = [0]
            seen_batters = set()

            for over in innings.get("overs", []):
                over_runs_total = 0
                over_bowler = None

                for ball in over.get("deliveries", []):
                    batter = ball["batter"]
                    bowler = ball["bowler"]
                    over_bowler = bowler
                    runs = ball["runs"]
                    extras = ball.get("extras", {})
                    total_runs = runs["total"]
                    over_runs_total += total_runs

                    if batter not in seen_batters:
                        seen_batters.add(batter)
                        bat_position_counter[0] += 1
                        if batter in players_stats:
                            players_stats[batter]["batting_position"] = bat_position_counter[0]

                    if batter in players_stats:
                        players_stats[batter]["balls_faced"] += 1
                        players_stats[batter]["runs"] += runs["batter"]
                        if runs["batter"] == 4:
                            players_stats[batter]["fours"] += 1
                        if runs["batter"] == 6:
                            players_stats[batter]["sixes"] += 1

                    if bowler in players_stats:
                        is_legal_ball = "wides" not in extras and "noballs" not in extras
                        if is_legal_ball:
                            players_stats[bowler]["balls_bowled"] += 1
                        byes_legbyes = extras.get("byes", 0) + extras.get("legbyes", 0)
                        players_stats[bowler]["runs_conceded"] += (total_runs - byes_legbyes)
                        players_stats[bowler]["wides"] += extras.get("wides", 0)
                        players_stats[bowler]["no_balls"] += extras.get("noballs", 0)
                        if total_runs == 0:
                            players_stats[bowler]["dot_balls"] += 1
                        if runs["batter"] == 4:
                            players_stats[bowler]["fours_conceded"] += 1
                        if runs["batter"] == 6:
                            players_stats[bowler]["sixes_conceded"] += 1

                    if "wickets" in ball:
                        for w in ball["wickets"]:
                            out_player = w["player_out"]
                            kind = w["kind"]

                            if out_player in players_stats:
                                players_stats[out_player]["dismissed"] = True
                                players_stats[out_player]["dismissal_type"] = kind

                            if bowler in players_stats and kind != "run out":
                                players_stats[bowler]["wickets"] += 1

                            fielders = [flr.get("name") for flr in w.get("fielders", []) if flr.get("name")]
                            for flr in fielders:
                                if flr not in players_stats:
                                    continue
                                if kind == "caught":
                                    players_stats[flr]["catches"] += 1
                                elif kind == "stumped":
                                    players_stats[flr]["stumpings"] += 1
                                elif kind == "run out":
                                    players_stats[flr]["run_outs"] += 1

                if over_bowler in players_stats and over_runs_total == 0:
                    players_stats[over_bowler]["maidens"] += 1

        all_rows.extend(players_stats.values())

    return pd.DataFrame(all_rows)

Test matches
Downloads Test data, builds the CSV, and shows a quick preview.

In [3]:
tests_df = build_player_match_dataset(
    "https://cricsheet.org/downloads/tests_male_json.zip", "tests_data"
)
tests_df.to_csv("data/test_player_match.csv", index=False)
print(tests_df.shape)
tests_df.head()

(19559, 29)


,match_id,player_id,player_name,team,opposition,date,format,venue,batting_position,runs,...,wides,no_balls,dot_balls,fours_conceded,sixes_conceded,catches,run_outs,stumpings,team_won,player_of_match
0,406189,4329fbb5,SR Watson,Australia,West Indies,2009-11-26,Test,"Brisbane Cricket Ground, Woolloongabba",1.0,0,...,0,0,82,11,0,3,0,0,True,False
1,406189,fdedb37c,SM Katich,Australia,West Indies,2009-11-26,Test,"Brisbane Cricket Ground, Woolloongabba",2.0,92,...,0,0,0,0,0,2,0,0,True,False
2,406189,7d415ea5,RT Ponting,Australia,West Indies,2009-11-26,Test,"Brisbane Cricket Ground, Woolloongabba",3.0,55,...,0,0,0,0,0,0,0,0,True,False
3,406189,48fd7349,MEK Hussey,Australia,West Indies,2009-11-26,Test,"Brisbane Cricket Ground, Woolloongabba",4.0,66,...,0,0,9,0,0,1,0,0,True,False
4,406189,f842c2cf,MJ Clarke,Australia,West Indies,2009-11-26,Test,"Brisbane Cricket Ground, Woolloongabba",5.0,41,...,0,0,0,0,0,1,0,0,True,False


In [4]:
import pandas as pd

df = pd.read_csv("data/test_player_match.csv")

summary = df.groupby(["player_id", "player_name"]).agg(
    format=("format", "first"),
    matches=("match_id", "nunique"),
    innings_batted=("balls_faced", lambda x: (x > 0).sum()),
    runs=("runs", "sum"),
    balls_faced=("balls_faced", "sum"),
    fours=("fours", "sum"),
    sixes=("sixes", "sum"),
    times_out=("dismissed", "sum"),
    innings_bowled=("balls_bowled", lambda x: (x > 0).sum()),
    balls_bowled=("balls_bowled", "sum"),
    runs_conceded=("runs_conceded", "sum"),
    wickets=("wickets", "sum"),
    maidens=("maidens", "sum"),
    catches=("catches", "sum"),
    run_outs=("run_outs", "sum"),
    stumpings=("stumpings", "sum"),
    player_of_match_count=("player_of_match", "sum"),
).reset_index()

# calculated fields
summary["batting_avg"] = (summary["runs"] / summary["times_out"].replace(0, pd.NA)).round(2)
summary["strike_rate"] = (summary["runs"] / summary["balls_faced"].replace(0, pd.NA) * 100).round(2)
summary["bowling_avg"] = (summary["runs_conceded"] / summary["wickets"].replace(0, pd.NA)).round(2)
summary["economy"] = (summary["runs_conceded"] / (summary["balls_bowled"].replace(0, pd.NA) / 6)).round(2)

summary.to_csv("data/test_player_summary.csv", index=False)
print(summary.shape)
summary.head()

(1063, 23)


,player_id,player_name,format,matches,innings_batted,runs,balls_faced,fours,sixes,times_out,...,wickets,maidens,catches,run_outs,stumpings,player_of_match_count,batting_avg,strike_rate,bowling_avg,economy
0,004c9e85,PJ Hughes,Test,26,26,1535,2871,199,11,26,...,0,0,15,0,0,1,59.038462,53.465691,<NA>,<NA>
1,00ea847a,MA Agarwal,Test,21,21,1488,2786,190,28,21,...,0,0,14,0,0,2,70.857143,53.409907,<NA>,<NA>
2,00ea9791,HP Tillakaratne,Test,2,2,70,180,9,0,2,...,0,0,0,0,0,0,35.0,38.888889,<NA>,<NA>
3,012829ff,JW Hastings,Test,1,1,52,89,5,2,1,...,1,3,1,0,0,0,52.0,58.426966,153.0,3.923077
4,0164b064,MG Neser,Test,5,5,131,258,19,1,5,...,22,21,1,0,0,0,26.2,50.775194,18.909091,3.208226


ODI


In [5]:
odis_df = build_player_match_dataset(
    "https://cricsheet.org/downloads/odis_male_json.zip", "odis_data"
)
odis_df.to_csv("data/odi_player_match.csv", index=False)
print(odis_df.shape)
odis_df.head()

(56540, 29)


,match_id,player_id,player_name,team,opposition,date,format,venue,batting_position,runs,...,wides,no_balls,dot_balls,fours_conceded,sixes_conceded,catches,run_outs,stumpings,team_won,player_of_match
0,426721,b8a55852,BB McCullum,New Zealand,Pakistan,2009-11-06,ODI,Sheikh Zayed Stadium,1.0,131,...,0,0,0,0,0,0,0,0,True,True
1,426721,48f177f5,AJ Redmond,New Zealand,Pakistan,2009-11-06,ODI,Sheikh Zayed Stadium,2.0,6,...,0,0,0,0,0,0,0,0,True,False
2,426721,2be41edb,MJ Guptill,New Zealand,Pakistan,2009-11-06,ODI,Sheikh Zayed Stadium,3.0,62,...,0,0,0,0,0,3,1,0,True,False
3,426721,b61a3e1a,LRPL Taylor,New Zealand,Pakistan,2009-11-06,ODI,Sheikh Zayed Stadium,4.0,0,...,0,0,0,0,0,2,0,0,True,False
4,426721,57efa3be,SB Styris,New Zealand,Pakistan,2009-11-06,ODI,Sheikh Zayed Stadium,5.0,9,...,1,0,11,0,1,0,0,0,True,False


In [6]:
df = pd.read_csv("data/odi_player_match.csv")

summary = df.groupby(["player_id", "player_name"]).agg(
    format=("format", "first"),
    matches=("match_id", "nunique"),
    innings_batted=("balls_faced", lambda x: (x > 0).sum()),
    runs=("runs", "sum"),
    balls_faced=("balls_faced", "sum"),
    fours=("fours", "sum"),
    sixes=("sixes", "sum"),
    times_out=("dismissed", "sum"),
    innings_bowled=("balls_bowled", lambda x: (x > 0).sum()),
    balls_bowled=("balls_bowled", "sum"),
    runs_conceded=("runs_conceded", "sum"),
    wickets=("wickets", "sum"),
    maidens=("maidens", "sum"),
    catches=("catches", "sum"),
    run_outs=("run_outs", "sum"),
    stumpings=("stumpings", "sum"),
    player_of_match_count=("player_of_match", "sum"),
).reset_index()

summary["batting_avg"] = (summary["runs"] / summary["times_out"].replace(0, pd.NA)).round(2)
summary["strike_rate"] = (summary["runs"] / summary["balls_faced"].replace(0, pd.NA) * 100).round(2)
summary["bowling_avg"] = (summary["runs_conceded"] / summary["wickets"].replace(0, pd.NA)).round(2)
summary["economy"] = (summary["runs_conceded"] / (summary["balls_bowled"].replace(0, pd.NA) / 6)).round(2)

summary.to_csv("data/odi_player_summary.csv", index=False)
print(summary.shape)
summary.head()

(1987, 23)


,player_id,player_name,format,matches,innings_batted,runs,balls_faced,fours,sixes,times_out,...,wickets,maidens,catches,run_outs,stumpings,player_of_match_count,batting_avg,strike_rate,bowling_avg,economy
0,004c9e85,PJ Hughes,ODI,25,24,826,1118,91,5,23,...,0,0,5,1,0,2,35.913043,73.881932,<NA>,<NA>
1,008589a4,CM Wright,ODI,7,6,81,145,4,1,5,...,8,9,1,0,0,0,16.2,55.862069,28.5,4.470588
2,00d69fa4,SO Tikolo,ODI,39,38,705,1097,75,6,35,...,26,6,16,0,0,0,20.142857,64.26618,34.692308,5.062675
3,00ea847a,MA Agarwal,ODI,5,5,86,91,12,1,5,...,0,0,2,0,0,0,17.2,94.505495,<NA>,10.0
4,00ea9791,HP Tillakaratne,ODI,16,13,332,562,28,0,11,...,0,0,10,0,0,0,30.181818,59.074733,<NA>,<NA>


T20

In [7]:
t20is_df = build_player_match_dataset(
    "https://cricsheet.org/downloads/t20s_male_json.zip", "t20is_data"
)
t20is_df.to_csv("data/t20i_player_match.csv", index=False)
print(t20is_df.shape)
t20is_df.head()

(76788, 29)


,match_id,player_id,player_name,team,opposition,date,format,venue,batting_position,runs,...,wides,no_balls,dot_balls,fours_conceded,sixes_conceded,catches,run_outs,stumpings,team_won,player_of_match
0,1354800,39e599c8,A McAuley,Isle of Man,Spain,2023-02-25,T20,La Manga Club Bottom Ground,1.0,29,...,0,0,0,0,0,0,0,0,False,False
1,1354800,b0aadf4f,N Knights,Isle of Man,Spain,2023-02-25,T20,La Manga Club Bottom Ground,2.0,1,...,0,0,0,0,0,1,0,0,False,False
2,1354800,c90346a0,D Jansen,Isle of Man,Spain,2023-02-25,T20,La Manga Club Bottom Ground,3.0,13,...,0,0,0,0,0,0,0,0,False,False
3,1354800,467c28f9,C Hartmann,Isle of Man,Spain,2023-02-25,T20,La Manga Club Bottom Ground,4.0,4,...,0,0,0,0,0,0,0,0,False,False
4,1354800,68019a70,EF Beard,Isle of Man,Spain,2023-02-25,T20,La Manga Club Bottom Ground,5.0,2,...,0,0,0,0,0,0,0,0,False,False


In [8]:
df = pd.read_csv("data/t20i_player_match.csv")

summary = df.groupby(["player_id", "player_name"]).agg(
    format=("format", "first"),
    matches=("match_id", "nunique"),
    innings_batted=("balls_faced", lambda x: (x > 0).sum()),
    runs=("runs", "sum"),
    balls_faced=("balls_faced", "sum"),
    fours=("fours", "sum"),
    sixes=("sixes", "sum"),
    times_out=("dismissed", "sum"),
    innings_bowled=("balls_bowled", lambda x: (x > 0).sum()),
    balls_bowled=("balls_bowled", "sum"),
    runs_conceded=("runs_conceded", "sum"),
    wickets=("wickets", "sum"),
    maidens=("maidens", "sum"),
    catches=("catches", "sum"),
    run_outs=("run_outs", "sum"),
    stumpings=("stumpings", "sum"),
    player_of_match_count=("player_of_match", "sum"),
).reset_index()

summary["batting_avg"] = (summary["runs"] / summary["times_out"].replace(0, pd.NA)).round(2)
summary["strike_rate"] = (summary["runs"] / summary["balls_faced"].replace(0, pd.NA) * 100).round(2)
summary["bowling_avg"] = (summary["runs_conceded"] / summary["wickets"].replace(0, pd.NA)).round(2)
summary["economy"] = (summary["runs_conceded"] / (summary["balls_bowled"].replace(0, pd.NA) / 6)).round(2)

summary.to_csv("data/t20i_player_summary.csv", index=False)
print(summary.shape)
summary.head()

(4695, 23)


,player_id,player_name,format,matches,innings_batted,runs,balls_faced,fours,sixes,times_out,...,wickets,maidens,catches,run_outs,stumpings,player_of_match_count,batting_avg,strike_rate,bowling_avg,economy
0,00029c30,M Mwamadi,T20,6,3,5,11,0,0,0,...,3,0,0,0,0,0,<NA>,45.454545,25.333333,7.354839
1,00075cb5,MCE Manalo,T20,1,1,12,16,0,0,1,...,0,0,0,0,0,0,12.0,75.0,<NA>,<NA>
2,000803b1,Het Patel,T20,2,1,1,9,0,0,0,...,0,0,1,0,0,0,<NA>,11.111111,<NA>,<NA>
3,00321fff,Mohammad Ghazanfar,T20,34,13,41,64,3,0,6,...,34,2,5,0,0,2,6.833333,64.0625,20.294118,6.854305
4,00467a76,S Bodha,T20,34,12,77,79,7,1,8,...,37,0,6,0,0,1,9.625,97.468354,22.810811,8.496644


Master CSV

In [9]:
import pandas as pd

tests_df = pd.read_csv("data/test_player_match.csv")
odis_df = pd.read_csv("data/odi_player_match.csv")
t20is_df = pd.read_csv("data/t20i_player_match.csv")

master_match = pd.concat([tests_df, odis_df, t20is_df], ignore_index=True)
master_match.to_csv("data/master_player_match.csv", index=False)

print(master_match.shape)



(152887, 29)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Feature engineering:**
Turning raw match stats into rolling form features the model can actually use.

Sort by player and date, so rolling stats are calculated in the right order.

In [10]:
df = pd.read_csv("data/master_player_match.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["player_id", "date"]).reset_index(drop=True)

print(df.shape)
df.head()

(152887, 29)


,match_id,player_id,player_name,team,opposition,date,format,venue,batting_position,runs,...,wides,no_balls,dot_balls,fours_conceded,sixes_conceded,catches,run_outs,stumpings,team_won,player_of_match
0,1283024,00029c30,M Mwamadi,Malawi,Uganda,2021-10-16,T20,Integrated Polytechnic Regional Centre,9.0,0,...,0,0,0,0,0,0,0,0,False,False
1,1283026,00029c30,M Mwamadi,Malawi,Swaziland,2021-10-17,T20,Integrated Polytechnic Regional Centre,NaN,0,...,1,0,5,0,0,0,0,0,True,False
2,1283029,00029c30,M Mwamadi,Malawi,Ghana,2021-10-19,T20,Gahanga International Cricket Stadium. Rwanda,9.0,3,...,0,0,1,1,2,0,0,0,False,False
3,1283033,00029c30,M Mwamadi,Malawi,Seychelles,2021-10-20,T20,Gahanga International Cricket Stadium. Rwanda,NaN,0,...,0,0,4,0,1,0,0,0,True,False
4,1283040,00029c30,M Mwamadi,Malawi,Rwanda,2021-10-22,T20,Gahanga International Cricket Stadium. Rwanda,4.0,2,...,3,0,12,0,2,0,0,0,True,False


Rolling batting form — average runs and strike rate over the last 5 and last 10 matches.

In [21]:
#step 2
def add_rolling_batting_features(df, window):
    grouped = df.groupby("player_id")

    df[f"runs_last{window}"] = grouped["runs"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    df[f"balls_faced_last{window}"] = grouped["balls_faced"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).sum()
    )
    runs_sum = grouped["runs"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).sum()
    )
    df[f"strike_rate_last{window}"] = (runs_sum / df[f"balls_faced_last{window}"] * 100).round(2)

    return df

df = add_rolling_batting_features(df, 5)
df = add_rolling_batting_features(df, 10)

df[["player_name", "date", "runs", "runs_last5", "strike_rate_last5", "runs_last10", "strike_rate_last10"]].head(15)
df.to_csv("data/master_player_match_features.csv", index=False)

Rolling bowling form — average wickets and economy over the last 5 and last 10 matches.

In [25]:
def add_rolling_bowling_features(df, window):
    grouped = df.groupby("player_id")

    df[f"wickets_last{window}"] = grouped["wickets"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    balls_bowled_sum = grouped["balls_bowled"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).sum()
    )
    runs_conceded_sum = grouped["runs_conceded"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).sum()
    )
    df[f"economy_last{window}"] = (runs_conceded_sum / (balls_bowled_sum / 6)).round(2)

    return df

df = add_rolling_bowling_features(df, 5)
df = add_rolling_bowling_features(df, 10)

df[["player_name", "date", "wickets", "wickets_last5", "economy_last5", "wickets_last10", "economy_last10"]].head(15)

,player_name,date,wickets,wickets_last5,economy_last5,wickets_last10,economy_last10
0,M Mwamadi,2021-10-16,0,NaN,NaN,NaN,NaN
1,M Mwamadi,2021-10-17,1,0.000000,NaN,0.000000,NaN
2,M Mwamadi,2021-10-19,0,0.500000,3.00,0.500000,3.00
3,M Mwamadi,2021-10-20,0,0.333333,8.70,0.333333,8.70
4,M Mwamadi,2021-10-22,2,0.250000,7.74,0.250000,7.74
5,M Mwamadi,2022-09-20,0,0.600000,7.35,0.600000,7.35
6,MCE Manalo,2025-07-09,0,NaN,NaN,NaN,NaN
7,Het Patel,2026-03-24,0,NaN,NaN,NaN,NaN
8,Het Patel,2026-03-25,0,0.000000,NaN,0.000000,NaN
9,Mohammad Ghazanfar,2019-10-06,0,NaN,NaN,NaN,NaN


Rolling fielding form average catches, run-outs, and stumpings over the last 5 and last 10 matches.

In [27]:
def add_rolling_fielding_features(df, window):
    grouped = df.groupby("player_id")

    df[f"catches_last{window}"] = grouped["catches"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    ).round(2)
    df[f"run_outs_last{window}"] = grouped["run_outs"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    ).round(2)
    df[f"stumpings_last{window}"] = grouped["stumpings"].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    ).round(2)

    return df

df = add_rolling_fielding_features(df, 5)
df = add_rolling_fielding_features(df, 10)

df[["player_name", "date", "catches", "catches_last5", "run_outs_last5", "stumpings_last5"]].head(15)
df.to_csv("data/master_player_match_features.csv", index=False)